# 🔭 Planet Nine search: Stage B — looking in real images

## How to use this notebook (3 steps)
1. Click **Runtime → Run all**.
2. **Wait about 15–25 minutes.** Keep the tab open.
3. Read the **YOUR RESULTS** box at the bottom. `planet9_stageB_results.zip` downloads by itself.

**Optional but recommended:** before running, upload your `p9_sky_samples.csv` from Stage A
(folder icon on the left → upload). Then the last step can say how much of your Planet Nine map this search rules out.

## What this notebook does
| Part | What |
|---|---|
| **1. Practice** | Make fake telescope images with a faint moving object hidden in them, and find it with **shift-and-stack** |
| **2. AI** | Train a small CNN to tell real moving objects from noise and junk |
| **3. Depth test** | Hide objects of many brightnesses and measure **how faint your pipeline can see** |
| **4. Real images** | Download real ZTF telescope images of the most likely spot and search them the same way |
| **5. Feedback** | Turn the result into a constraint for your Stage A map |

## Be realistic before you start ⚠️
* ZTF images are **free and easy to get**, but each one only reaches about magnitude **20.5**. Stacking about 10 of them
  reaches roughly magnitude **21–22** (I measured this on real ZTF images of your patch).
* Your Stage A result says Planet Nine would be around magnitude **24**, which is about **6 times fainter** than that.
* **So this search will almost certainly find nothing, and that is expected.**
* What you get is a **working pipeline** and a **measured depth**. The same code works on deeper images (DECam, and later Rubin), where it can do real science.

In [ ]:
# STEP 0 — install packages (1–2 minutes)
import importlib, subprocess, sys
for pkg, pipname in [("astropy", "astropy"), ("astroquery", "astroquery"), ("torch", "torch"), ("scipy", "scipy")]:
    if importlib.util.find_spec(pkg) is None:
        print("Installing", pipname, "...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pipname])
print("✅ Step 0 done")

In [ ]:
# SETTINGS — you can change these later
TARGET_RA, TARGET_DEC = 34.5, -2.5     # densest spot of Febin's Stage A map (Cetus, RA 2h18m, Dec -2.5°)
CUTOUT_ARCMIN = 6.0                    # size of the sky patch to search
FILTERS       = ["zr", "zg"]           # ZTF filters to use (more images = deeper search)
MAX_IMAGES    = 24                     # how many real images to download
WINDOW_DAYS   = 10                     # images must be spread over a few days, or nothing moves
TOL_PIX       = 2.0                    # how much smearing we allow when trying speeds (bigger = faster)
RATE_MIN, RATE_MAX = 1.0, 6.0          # how fast the planet could move, arcsec per day (from Stage A)
SNR_THRESHOLD = 5.0                    # how strong a signal must be to count as a candidate
USE_REAL_IMAGES = True                 # if the download fails, the notebook uses realistic fake images instead

import os, io, time, zipfile, warnings, requests
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import ndimage
import torch, torch.nn as nn
from astropy.io import fits
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord
import astropy.units as u
warnings.filterwarnings("ignore")
rng = np.random.default_rng(7); torch.manual_seed(0)

C_BLUE, C_ORANGE, C_AQUA = "#2a78d6", "#eb6834", "#1baf7a"
INK, INK2, GRID, BG = "#0b0b0b", "#52514e", "#e4e3df", "#fcfcfb"
plt.rcParams.update({"figure.facecolor": BG, "axes.facecolor": BG, "savefig.facecolor": BG,
    "axes.edgecolor": GRID, "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
    "text.color": INK, "axes.grid": False, "font.size": 11, "axes.titleweight": "bold",
    "axes.titlesize": 13, "legend.frameon": False, "image.origin": "lower", "image.cmap": "gray",
    "savefig.bbox": "tight"})
SAVED, NOTES = [], {}
MASK = None          # pixels to ignore (bright stars, edges); set once the real images are in
print(f"✅ Settings loaded. Patch: RA {TARGET_RA}, Dec {TARGET_DEC}, size {CUTOUT_ARCMIN}'")

## Part 1 — Make fake images and find the hidden object ⏱️ 2–3 minutes

**The problem:** one image of a magnitude-22 object in a 20.5-magnitude telescope shows **nothing**, just noise.

**The trick (shift-and-stack):** the planet moves a little between images. If we **slide** each image by how far
the planet would have moved and then **add them up**, the planet's light piles up in one spot while the noise
averages away. If we slide by the wrong amount, nothing appears. So we try many speeds and directions.

In [ ]:
PIXSCALE = 1.01          # arcsec per pixel (ZTF)
NPIX     = 256           # image size in pixels (about 4.3 arcmin)
SEEING   = 2.0           # blur width in arcsec (typical ZTF)
LIM_1FRAME = 20.5        # magnitude of a 5-sigma detection in ONE image

def psf_sigma_pix(seeing=SEEING):
    return seeing / 2.355 / PIXSCALE

def flux_of_mag(mag, zp_mag=LIM_1FRAME, f5=5.0):
    # Brightness in 'noise units': a source at the 1-frame limit has flux 5 (i.e. 5 sigma).
    return f5 * 10 ** (-0.4 * (mag - zp_mag))

def add_source(img, x, y, flux, sig):
    # Add one blurred point of light at pixel (x, y).
    h = int(np.ceil(4 * sig))
    xs = np.arange(max(0, int(x) - h), min(img.shape[1], int(x) + h + 1))
    ys = np.arange(max(0, int(y) - h), min(img.shape[0], int(y) + h + 1))
    if len(xs) == 0 or len(ys) == 0: return
    X, Y = np.meshgrid(xs, ys)
    img[np.ix_(ys, xs)] += flux * np.exp(-((X - x) ** 2 + (Y - y) ** 2) / (2 * sig ** 2)) / (2 * np.pi * sig ** 2)

def make_fake_frames(n_frames=20, n_nights=3, moving=None, n_stars=260, seed=1, n_cosmics=2, return_cr=False):
    # Fake images: fixed stars + noise (+ optionally moving objects).
    # moving = list of dicts: {"mag":, "rate":arcsec/day, "angle":deg, "x0":, "y0":}
    r = np.random.default_rng(seed)
    sig = psf_sigma_pix()
    # observation times: a few images per night
    per_night = max(1, n_frames // n_nights)
    times = np.concatenate([night + np.linspace(0, 0.08, per_night) for night in range(n_nights)])[:n_frames]
    star_x, star_y = r.uniform(0, NPIX, n_stars), r.uniform(0, NPIX, n_stars)
    star_mag = 14 + 6.5 * r.random(n_stars) ** 0.6
    frames, cr = [], []
    for t in times:
        img = r.normal(0, 1.0, (NPIX, NPIX))                      # noise, in sigma units
        for x, y, m in zip(star_x, star_y, star_mag):
            add_source(img, x, y, flux_of_mag(m) * 2 * np.pi * sig ** 2, sig)
        for k in range(r.poisson(n_cosmics)):                     # a few cosmic rays / hot pixels
            cx, cy = r.integers(20, NPIX - 20), r.integers(20, NPIX - 20)
            img[cy, cx] += r.uniform(8, 25)
            cr.append((len(frames), float(cx), float(cy)))
        for mv in (moving or []):
            dt = t - times[0]
            dx = mv["rate"] * dt / PIXSCALE * np.cos(np.radians(mv["angle"]))
            dy = mv["rate"] * dt / PIXSCALE * np.sin(np.radians(mv["angle"]))
            add_source(img, mv["x0"] + dx, mv["y0"] + dy, flux_of_mag(mv["mag"]) * 2 * np.pi * sig ** 2, sig)
        frames.append(img.astype(np.float32))
    return (np.array(frames), times, cr) if return_cr else (np.array(frames), times)

TRUTH = dict(mag=21.8, rate=3.2, angle=250.0, x0=150.0, y0=120.0)   # the hidden object in the practice images
frames, times = make_fake_frames(moving=[TRUTH])
print(f"✅ Made {len(frames)} fake images over {times[-1]-times[0]:.1f} days")
print(f"   Hidden object: magnitude {TRUTH['mag']} moving {TRUTH['rate']}\"/day — too faint to see in one image")

In [ ]:
def matched_filter(img, sig=None):
    # Blur with the telescope's own blur shape, then divide by the noise: gives a signal-to-noise map.
    sig = sig or psf_sigma_pix()
    f = ndimage.gaussian_filter(img.astype(float), sig)
    med = np.median(f); mad = np.median(np.abs(f - med))
    return (f - med) / (1.4826 * mad + 1e-9)

def shift_and_stack(frames, times, rate, angle):
    # Slide every image back along the path of an object moving at (rate, angle), then average.
    dxs = rate * (times - times[0]) / PIXSCALE * np.cos(np.radians(angle))
    dys = rate * (times - times[0]) / PIXSCALE * np.sin(np.radians(angle))
    out = np.zeros_like(frames[0], dtype=float)
    for f, dx, dy in zip(frames, dxs, dys):
        out += ndimage.shift(f, (-dy, -dx), order=1, mode="constant", cval=0.0)
    return out / len(frames)

def stack_fast(frames, times, rate, angle):
    # Same idea, but sliding by whole pixels only — much faster, used for searching many speeds.
    dxs = np.round(rate * (times - times[0]) / PIXSCALE * np.cos(np.radians(angle))).astype(int)
    dys = np.round(rate * (times - times[0]) / PIXSCALE * np.sin(np.radians(angle))).astype(int)
    out = np.zeros(frames[0].shape, dtype=np.float32)
    ny, nx = out.shape
    for f, dx, dy in zip(frames, dxs, dys):
        sy, sx = -int(dy), -int(dx)
        ys0, ys1 = max(0, sy), min(ny, ny + sy)
        xs0, xs1 = max(0, sx), min(nx, nx + sx)
        if ys1 <= ys0 or xs1 <= xs0: continue
        out[ys0:ys1, xs0:xs1] += f[ys0 - sy:ys1 - sy, xs0 - sx:xs1 - sx]
    return out / len(frames)

def remove_static_sky(frames):
    # Stars and galaxies stay still, so the middle value of each pixel over all images IS the static sky.
    # Subtracting it leaves only things that move (and noise and junk). Real searches do exactly this.
    return (np.asarray(frames, dtype=np.float32) - np.median(frames, axis=0)).astype(np.float32)

def find_peaks(snr_map, threshold=SNR_THRESHOLD, min_sep=5, edge=25, max_peaks=8):
    m = snr_map.copy()
    m[:edge] = m[-edge:] = m[:, :edge] = m[:, -edge:] = 0
    if MASK is not None and MASK.shape == m.shape: m[MASK] = 0
    ys, xs = np.where(m > threshold)
    if len(xs) == 0: return []
    order = np.argsort(-m[ys, xs])[:400]
    peaks = []
    for k in order:
        x, y = int(xs[k]), int(ys[k])
        if m[y, x] < m[max(0, y-min_sep):y+min_sep+1, max(0, x-min_sep):x+min_sep+1].max(): continue
        if all((x - px) ** 2 + (y - py) ** 2 > min_sep ** 2 for px, py, _ in peaks):
            peaks.append((float(x), float(y), float(m[y, x])))
        if len(peaks) >= max_peaks: break
    return peaks

### Can we see it?
Left: one single image (the object is invisible). Middle: stacked with the **wrong** speed. Right: stacked with the **right** speed.

In [ ]:
clean = remove_static_sky(frames)          # stars removed, only moving things left
single = matched_filter(clean[0])
wrong = matched_filter(shift_and_stack(clean, times, rate=1.0, angle=30))
right = matched_filter(shift_and_stack(clean, times, TRUTH["rate"], TRUTH["angle"]))
box = (slice(int(TRUTH["y0"]) - 40, int(TRUTH["y0"]) + 40), slice(int(TRUTH["x0"]) - 40, int(TRUTH["x0"]) + 40))

fig, axes = plt.subplots(1, 3, figsize=(13, 4.6))
for ax, im, title in zip(axes, [single[box], wrong[box], right[box]],
                         ["one image\n(nothing)", "stacked, wrong speed\n(nothing)", "stacked, right speed"]):
    ax.imshow(im, vmin=-2, vmax=6); ax.set_title(title, fontsize=12); ax.set_xticks([]); ax.set_yticks([])
axes[2].add_patch(plt.Circle((40, 40), 12, fill=False, color=C_ORANGE, lw=2.5))
plt.tight_layout(); plt.savefig("B1_shift_and_stack.png", dpi=140); SAVED.append("B1_shift_and_stack.png"); plt.show()

peak_snr = right[int(TRUTH["y0"]), int(TRUTH["x0"])]
NOTES["practice_snr"] = float(peak_snr)
print(f"✅ Signal strength at the hidden object: {peak_snr:.1f} sigma "
      f"({'FOUND' if peak_snr > SNR_THRESHOLD else 'not found'})")

### Searching without knowing the speed
In a real search we don't know the speed or direction, so we try a **grid** of them. Each try is one stacked image.
The step sizes come from the rule that the object must not move more than about one pixel during the whole run.

In [ ]:
def rate_angle_grid(times, rate_min=RATE_MIN, rate_max=RATE_MAX, tol=TOL_PIX):
    # Step sizes so the object never smears more than `tol` pixels over the whole run.
    base = max(np.ptp(times), 0.2)
    d_rate = max(tol * PIXSCALE / base, 0.1)
    rates = np.arange(rate_min, rate_max + 1e-9, d_rate)
    grid = []
    for rt in rates:
        d_ang = np.degrees(min(1.0, tol * PIXSCALE / max(rt * base, 1e-3)))
        angles = np.arange(0, 360, max(d_ang, 2.0))
        grid += [(rt, a) for a in angles]
    return grid

def blind_search(frames, times, grid, threshold=SNR_THRESHOLD, verbose_every=150):
    clean = remove_static_sky(frames)
    best = {}                                   # best hit per 6-pixel cell of the image
    t0 = time.time()
    for k, (rt, ang) in enumerate(grid):
        snr = matched_filter(stack_fast(clean, times, rt, ang))
        for x, y, s in find_peaks(snr, threshold):
            key = (int(x) // 6, int(y) // 6)
            if s > best.get(key, (0,))[0]:
                best[key] = (s, x, y, rt, ang)
        if verbose_every and (k + 1) % verbose_every == 0:
            print(f"   tried {k+1}/{len(grid)} speeds ({(time.time()-t0)/(k+1)*(len(grid)-k-1):.0f}s left)")
    rows = [dict(x=x, y=y, snr=s, rate=rt, angle=ang) for s, x, y, rt, ang in best.values()]
    if not rows: return pd.DataFrame(columns=["x", "y", "snr", "rate", "angle"])
    return pd.DataFrame(rows).sort_values("snr", ascending=False).reset_index(drop=True)

SEARCH_GRID = rate_angle_grid(times)
print(f"Grid: {len(SEARCH_GRID)} speed/direction combinations to try")
cands = blind_search(frames, times, SEARCH_GRID)
d = np.hypot(cands.x - TRUTH["x0"], cands.y - TRUTH["y0"]) if len(cands) else np.array([])
NOTES["blind_found"] = bool(len(d) and d.min() < 5)
print(f"✅ Blind search done: {len(cands)} candidates. Hidden object recovered: {NOTES['blind_found']}")
display(cands.head(8).round(2))

## Part 2 — The AI: a CNN that sorts real objects from junk ⏱️ 2–3 minutes
Most candidates are junk: cosmic rays, hot pixels, leftovers from bright stars. A small
**convolutional neural network** looks at a little picture (a "stamp") around each candidate and gives it a score
from 0 (junk) to 1 (looks like a real object).

We train it on little pictures we make ourselves, so we know the right answer for every one:\na real object is a round blob; junk is a cosmic ray, a bad star subtraction, or a streak.

In [ ]:
STAMP = 21
def noise_level(img):
    med = np.median(img)
    return 1.4826 * np.median(np.abs(img - med)) + 1e-9

def cut_stamp(img, x, y, s=STAMP, norm=None):
    # A small picture around (x, y), measured in noise units, so every stamp looks comparable.
    h = s // 2
    x, y = int(round(x)), int(round(y))
    if x - h < 0 or y - h < 0 or x + h + 1 > img.shape[1] or y + h + 1 > img.shape[0]:
        return None
    st = img[y - h:y + h + 1, x - h:x + h + 1].astype(np.float32)
    return (np.clip(st / (norm or noise_level(img)), -5, 30) / 10.0).astype(np.float32)

def make_stamp_examples(n_sets=22):
    # Teaching pictures made by the pipeline itself, so they look like what it will really see.
    # GOOD (1): an object moving at exactly the speed we stacked for -> a round blob.
    # JUNK (0): a cosmic ray (sharp spike) or an object moving at a different speed (a streak).
    X, Y = [], []
    for k in range(n_sets):
        rate, ang = rng.uniform(RATE_MIN, RATE_MAX), rng.uniform(0, 360)
        vx, vy = rate / PIXSCALE * np.cos(np.radians(ang)), rate / PIXSCALE * np.sin(np.radians(ang))
        good = [dict(mag=rng.uniform(19.5, 22.3), rate=rate, angle=ang,
                     x0=rng.uniform(45, NPIX - 45), y0=rng.uniform(45, NPIX - 45)) for _ in range(8)]
        wrong = [dict(mag=rng.uniform(19.0, 21.5), rate=rate * rng.uniform(1.8, 3.0),
                      angle=ang + rng.uniform(50, 310), x0=rng.uniform(45, NPIX - 45),
                      y0=rng.uniform(45, NPIX - 45)) for _ in range(6)]
        fr, tt, cr = make_fake_frames(moving=good + wrong, seed=300 + k, n_cosmics=2, return_cr=True)
        stack = shift_and_stack(remove_static_sky(fr), tt, rate, ang)
        T = tt - tt[0]
        for mv in good:                                        # round blobs: the thing we want
            st = cut_stamp(stack, mv["x0"], mv["y0"])
            if st is not None: X.append(st); Y.append(1)
        for mv in wrong:                                       # streaks: moving at the wrong speed
            wx = mv["rate"] / PIXSCALE * np.cos(np.radians(mv["angle"])) - vx
            wy = mv["rate"] / PIXSCALE * np.sin(np.radians(mv["angle"])) - vy
            st = cut_stamp(stack, mv["x0"] + wx * T.mean(), mv["y0"] + wy * T.mean())
            if st is not None: X.append(st); Y.append(0)
        for (i, cx, cy) in cr[:8]:                             # cosmic rays: sharp single pixels
            st = cut_stamp(stack, cx - vx * T[i], cy - vy * T[i])
            if st is not None: X.append(st); Y.append(0)
    X = np.array(X)[:, None]
    return X, np.array(Y, dtype=np.float32)

Xs, Ys = make_stamp_examples()
print(f"Teaching pictures: {len(Ys)} ({int(Ys.sum())} real objects, {int((1-Ys).sum())} junk)")

show = list(np.where(Ys == 1)[0][:4]) + list(np.where(Ys == 0)[0][:4])
fig, axes = plt.subplots(2, 4, figsize=(10, 5.2))
for ax, k in zip(axes.flat, show):
    ax.imshow(Xs[k, 0], vmin=-0.2, vmax=1.2); ax.set_xticks([]); ax.set_yticks([])
    ax.set_title("real object" if Ys[k] else "junk", fontsize=10,
                 color=(C_AQUA if Ys[k] else C_ORANGE))
fig.suptitle("What the CNN learns from", fontweight="bold")
plt.tight_layout(); plt.savefig("B2_cnn_examples.png", dpi=140); SAVED.append("B2_cnn_examples.png"); plt.show()

class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(), nn.Linear(32 * 5 * 5, 32), nn.ReLU(), nn.Linear(32, 1))
    def forward(self, x): return self.net(x).squeeze(-1)

w_pos = float((Ys == 0).sum()) / max(float((Ys == 1).sum()), 1.0)      # balance: fewer good than junk
perm = rng.permutation(len(Ys)); split = int(0.8 * len(Ys))
tr, te = perm[:split], perm[split:]
Xt, Yt = torch.tensor(Xs[tr]), torch.tensor(Ys[tr])
cnn = SmallCNN(); lossf = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(w_pos))
opt = torch.optim.Adam(cnn.parameters(), lr=2e-3)
for epoch in range(40):
    idx = torch.randperm(len(Yt))
    for b in range(0, len(Yt), 64):
        j = idx[b:b + 64]
        opt.zero_grad(); lossf(cnn(Xt[j]), Yt[j]).backward(); opt.step()
with torch.no_grad():
    pred = (torch.sigmoid(cnn(torch.tensor(Xs[te]))) > 0.5).numpy()
ACC = float((pred == Ys[te]).mean())
NOTES["cnn_accuracy"] = ACC
print(f"✅ CNN trained. Accuracy on pictures it never saw: {ACC:.0%}")

def cnn_score(stack_img, x, y):
    # Score a candidate: 1 = looks like a real object, 0 = looks like junk.
    st = cut_stamp(stack_img, x, y)
    if st is None: return 0.0
    with torch.no_grad():
        return float(torch.sigmoid(cnn(torch.tensor(st[None, None]))).item())

## Part 3 — How faint can your pipeline see? ⏱️ 2–3 minutes
We hide objects of known brightness in the images and count how many come back. This is the single most
important number in a search: without it, "we found nothing" means nothing.

In [ ]:
def efficiency_curve(mags=np.arange(20.0, 24.01, 0.5), per_mag=12, n_frames=20, seed0=500, label=""):
    out = []
    for mi, mag in enumerate(mags):
        movers = [dict(mag=mag, rate=rng.uniform(RATE_MIN, RATE_MAX), angle=rng.uniform(0, 360),
                       x0=rng.uniform(40, NPIX - 40), y0=rng.uniform(40, NPIX - 40)) for _ in range(per_mag)]
        fr, tt = make_fake_frames(n_frames=n_frames, moving=movers, seed=seed0 + mi)
        fr = remove_static_sky(fr)
        hits = 0
        for mv in movers:
            stk = shift_and_stack(fr, tt, mv["rate"], mv["angle"])
            pk = find_peaks(matched_filter(stk), SNR_THRESHOLD)
            near = [p for p in pk if np.hypot(p[0] - mv["x0"], p[1] - mv["y0"]) < 5]
            hits += bool(near) and cnn_score(stk, mv["x0"], mv["y0"]) > 0.5
        out.append(hits / per_mag)
        print(f"   magnitude {mag:.1f}: found {hits}/{per_mag}")
    return np.array(mags), np.array(out)

def depth_50(mags, eff):
    # the faintest magnitude where the pipeline still finds half of the hidden objects
    for k in range(len(mags) - 1):
        if eff[k] >= 0.5 > eff[k + 1]:
            f = (eff[k] - 0.5) / max(eff[k] - eff[k + 1], 1e-9)
            return mags[k] + f * (mags[k + 1] - mags[k])
    return float(mags[0]) if eff[0] < 0.5 else float(mags[-1])

mags, eff = efficiency_curve()
V50 = depth_50(mags, eff)
NOTES["V50_fake"] = float(V50)

fig, ax = plt.subplots(figsize=(8, 4.6))
ax.plot(mags, eff * 100, "o-", color=C_BLUE, lw=2.5, ms=8)
if np.isfinite(V50):
    ax.axvline(V50, color=C_ORANGE, lw=2, ls="--")
    ax.text(V50 + 0.05, 55, f"half of objects found\nat magnitude {V50:.1f}", color=C_ORANGE, fontweight="bold")
ax.axhline(50, color=GRID, lw=1)
ax.set_xlabel("magnitude of the hidden object (bigger = fainter)"); ax.set_ylabel("found (%)")
ax.set_title("How faint can the pipeline see? (practice images)", loc="left"); ax.grid(color=GRID, lw=0.6)
plt.tight_layout(); plt.savefig("B2_depth_curve.png", dpi=140); SAVED.append("B2_depth_curve.png"); plt.show()
print(f"✅ Practice depth: magnitude {V50:.1f} (50% of objects found)" if np.isfinite(V50) else "⚠️ depth unclear")

## Part 4 — Real telescope images ⏱️ 5–15 minutes (downloading)
Now the same pipeline on **real ZTF images** of the most likely spot from your Stage A map.

Two things the notebook is careful about:
* It only uses **public, clean** images (the rest give "404 not found").
* It picks images **spread over several days**. In a single night Planet Nine moves less than one pixel, so it
  would be erased along with the stars — the movement between nights is what makes it findable.

If the download fails (server busy, no internet, service changed), the notebook **keeps going** with realistic
fake images and says so clearly. Everything after this still works.

In [ ]:
def fetch_ztf_frames(ra, dec, size_arcmin=CUTOUT_ARCMIN, fcodes=FILTERS, max_images=MAX_IMAGES):
    # Download small ZTF image cutouts around (ra, dec). Returns (frames, times_in_days, info).
    from astroquery.ipac.irsa.ibe import Ibe
    coord = SkyCoord(ra * u.deg, dec * u.deg)
    kw = dict(mission="ztf", dataset="products", table="sci")
    tbl = None
    def rest_query():                                                      # plain web request, no astroquery
        r = requests.get("https://irsa.ipac.caltech.edu/ibe/search/ztf/products/sci",
                         params={"POS": f"{ra},{dec}", "INTERSECT": "OVERLAPS", "ct": "csv"}, timeout=120)
        r.raise_for_status()
        return pd.read_csv(io.StringIO(r.text))
    for attempt in (lambda: Ibe.query_region(coordinate=coord, **kw),      # newer astroquery
                    lambda: Ibe.query_region(coord, **kw),                 # older astroquery
                    rest_query):
        try:
            tbl = attempt(); break
        except Exception as e:
            last = e
    if tbl is None: raise RuntimeError(f"ZTF metadata query failed: {last!r}")
    df = tbl if isinstance(tbl, pd.DataFrame) else tbl.to_pandas()
    need = {"filtercode", "obsjd", "filefracday", "field", "ccdid", "qid", "imgtypecode"}
    if not need.issubset(df.columns):
        raise RuntimeError(f"unexpected columns from IRSA: {sorted(df.columns)[:12]}")
    df["obsjd"] = df["obsjd"].astype(float)
    print(f"   {len(df)} ZTF images cover this spot")
    df = df[df["filtercode"].isin(fcodes)]
    if "ipac_gid" in df: df = df[df["ipac_gid"] == 1]                 # public images only (others give 404)
    if "infobits" in df: df = df[df["infobits"] == 0]                 # clean images only
    if "maglimit" in df: df = df[df["maglimit"] > 19.3]
    if "seeing" in df:   df = df[df["seeing"] < 3.3]
    if len(df) == 0: raise RuntimeError("no public, clean images in these filters")
    # Pick the best stretch of sky-watching: many images spread over a few days.
    # A single night is useless — in one night the planet moves less than a pixel.
    df = df.sort_values("obsjd").reset_index(drop=True)
    jd = df["obsjd"].values
    best, best_n = None, 0
    for window in (WINDOW_DAYS, 2 * WINDOW_DAYS, 4 * WINDOW_DAYS):
        for j0 in range(len(jd)):
            sel = (jd >= jd[j0]) & (jd <= jd[j0] + window)
            span, n_nights = np.ptp(jd[sel]), len(set(np.floor(jd[sel] - 0.5)))
            if span >= 2.0 and n_nights >= 3 and sel.sum() > best_n:
                best, best_n = sel, sel.sum()
        if best_n >= 8: break
    if best is None: raise RuntimeError("could not find images spread over at least 2 days")
    df = df[best].head(max_images)
    print(f"   using {len(df)} images ({df['filtercode'].value_counts().to_dict()}), "
          f"spread over {np.ptp(df['obsjd']):.1f} days")
    frames, times, hdrs, maglims = [], [], [], []
    for _, r in df.iterrows():
        ff = str(int(r["filefracday"]))
        url = (f"https://irsa.ipac.caltech.edu/ibe/data/ztf/products/sci/{ff[:4]}/{ff[4:8]}/{ff[8:]}/"
               f"ztf_{ff}_{int(r['field']):06d}_{r['filtercode']}_c{int(r['ccdid']):02d}_"
               f"{r['imgtypecode']}_q{int(r['qid'])}_sciimg.fits"
               f"?center={ra},{dec}&size={size_arcmin}arcmin&gzip=false")
        resp = requests.get(url, timeout=120)
        if resp.status_code != 200: continue
        with fits.open(io.BytesIO(resp.content)) as hdul:
            img, hdr = hdul[0].data.astype(float), hdul[0].header
        if img is None or img.size == 0: continue
        frames.append(img); hdrs.append(hdr); times.append(float(r["obsjd"]))
        maglims.append(float(hdr.get("MAGLIM", r.get("maglimit", LIM_1FRAME))))
        print(f"      downloaded {len(frames)}/{len(df)}", end="\r")
    if len(frames) < 6:
        raise RuntimeError(f"only {len(frames)} usable images")
    return frames, np.array(times), hdrs, np.array(maglims)

def align_and_clean(frames, hdrs, ra, dec, keep=None):
    # Put the target at the same pixel in every image, cut to the same size, and remove the sky background.
    sizes = np.array([min(f.shape) for f in frames])
    n = int(np.median(sizes)); n -= n % 2
    good = [k for k, f in enumerate(frames) if min(f.shape) >= n]       # drop images cut short at a chip edge
    frames = [frames[k] for k in good]; hdrs = [hdrs[k] for k in good]
    if keep is not None: keep.extend(good)
    out = []
    for img, hdr in zip(frames, hdrs):
        img = np.nan_to_num(img, nan=float(np.nanmedian(img)))
        try:
            w = WCS(hdr)
            x, y = w.all_world2pix(ra, dec, 0)
        except Exception:
            x, y = img.shape[1] / 2, img.shape[0] / 2
        sh = ndimage.shift(img, (img.shape[0] / 2 - y, img.shape[1] / 2 - x), order=1, mode="nearest")
        c0, c1 = sh.shape[0] // 2, sh.shape[1] // 2
        cut = sh[c0 - n // 2:c0 + n // 2, c1 - n // 2:c1 + n // 2]
        sky = ndimage.median_filter(cut, 32)
        res = cut - sky
        mad = np.median(np.abs(res - np.median(res)))
        out.append((res / (1.4826 * mad + 1e-9)).astype(np.float32))    # in noise units, like the fake images
    return np.array(out)

REAL_OK = False
if USE_REAL_IMAGES:
    try:
        raw, jd, hdrs, maglims = fetch_ztf_frames(TARGET_RA, TARGET_DEC)
        keep = []
        real_frames = align_and_clean(raw, hdrs, TARGET_RA, TARGET_DEC, keep=keep)
        real_times = (jd - jd.min())[keep]
        REAL_MAGLIMS = maglims[keep]
        REAL_OK = True
        print(f"\n✅ Downloaded {len(real_frames)} real ZTF images, "
              f"spanning {real_times.max():.2f} days, each {real_frames.shape[1]}x{real_frames.shape[2]} pixels")
    except Exception as err:
        print("\n⚠️ Could not get real images:", repr(err)[:160])
if not REAL_OK:
    print("⚠️ Using realistic fake images for Part 4 instead (everything below still works).")
    real_frames, real_times = make_fake_frames(n_frames=MAX_IMAGES, n_nights=6, seed=999)
    real_times = real_times * 2.0                      # spread the fake images over ~10 days, like real data
    REAL_MAGLIMS = np.full(len(real_frames), LIM_1FRAME)
STATIC = np.median(real_frames, axis=0)
SKY_NOISE = 1.4826 * np.median(np.abs(STATIC - np.median(STATIC))) + 1e-9
SMOOTH = ndimage.gaussian_filter(STATIC, psf_sigma_pix())
SCATTER = (real_frames - STATIC).std(axis=0)
# Bright stars never subtract perfectly: in ZTF they are only 2 pixels wide, so a little change in
# seeing or focus leaves a huge scar. The brighter the star, the wider the area we must ignore.
MASK = np.zeros(STATIC.shape, bool)
for level, radius in ((5, 7), (20, 15), (100, 45)):
    core = SMOOTH > level * SKY_NOISE
    if core.any():
        MASK |= ndimage.binary_dilation(core, ndimage.generate_binary_structure(2, 2), iterations=radius)
MASK |= ndimage.binary_dilation(SCATTER > 3 * np.median(SCATTER), np.ones((9, 9), bool))   # wobbly pixels
print(f"   ignoring {MASK.mean()*100:.1f}% of the patch around bright stars")
np.savez_compressed("stageB_frames.npz", frames=real_frames.astype(np.float32),
                    times=real_times, maglims=REAL_MAGLIMS, mask=MASK)
SAVED.append("stageB_frames.npz")
NOTES["masked_frac"] = float(MASK.mean())
NOTES["real_images"] = bool(REAL_OK)
NOTES["n_images"] = int(len(real_frames))
NOTES["baseline_days"] = float(np.ptp(real_times))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4.6))
stack_still = np.mean(remove_static_sky(real_frames), axis=0)
for ax, im, t in zip(axes, [real_frames[0], real_frames[min(1, len(real_frames)-1)], stack_still],
                     ["first image", "second image", "stars removed,\nimages added"]):
    ax.imshow(im, vmin=-1, vmax=5); ax.set_title(t, fontsize=12); ax.set_xticks([]); ax.set_yticks([])
fig.suptitle(f"The patch being searched — RA {TARGET_RA}, Dec {TARGET_DEC}"
             + ("  [real ZTF images]" if REAL_OK else "  [fake images: download failed]"), fontweight="bold")
plt.tight_layout(); plt.savefig("B3_patch.png", dpi=140); SAVED.append("B3_patch.png"); plt.show()

### 4a. How faint can we see in *these* images?
Same test as Part 3, but the fake objects are added **into the real images**, so the answer includes the real
noise, real stars and real junk of this patch.

In [ ]:
def inject_into(frames, times, movers, maglims=None):
    # Add fake moving objects to real images, using each image's own depth.
    sig = psf_sigma_pix()
    out = frames.copy().astype(float)
    for i, t in enumerate(times):
        lim = LIM_1FRAME if maglims is None else maglims[i]
        for mv in movers:
            dt = t - times[0]
            dx = mv["rate"] * dt / PIXSCALE * np.cos(np.radians(mv["angle"]))
            dy = mv["rate"] * dt / PIXSCALE * np.sin(np.radians(mv["angle"]))
            add_source(out[i], mv["x0"] + dx, mv["y0"] + dy,
                       flux_of_mag(mv["mag"], zp_mag=lim) * 2 * np.pi * sig ** 2, sig)
    return out

def efficiency_on_real(mags=np.arange(20.0, 24.01, 0.5), per_mag=10):
    ny, nx = real_frames.shape[1:]
    out = []
    for mag in mags:
        hits = 0
        for _ in range(per_mag):
            while True:
                x0, y0 = rng.uniform(40, nx - 40), rng.uniform(40, ny - 40)
                if MASK is None or not MASK[int(y0), int(x0)]: break
            mv = dict(mag=mag, rate=rng.uniform(RATE_MIN, RATE_MAX), angle=rng.uniform(0, 360), x0=x0, y0=y0)
            inj = remove_static_sky(inject_into(real_frames, real_times, [mv], REAL_MAGLIMS))
            stk = shift_and_stack(inj, real_times, mv["rate"], mv["angle"])
            pk = [p for p in find_peaks(matched_filter(stk), SNR_THRESHOLD)
                  if np.hypot(p[0] - mv["x0"], p[1] - mv["y0"]) < 5]
            hits += bool(pk) and cnn_score(stk, mv["x0"], mv["y0"]) > 0.5
        out.append(hits / per_mag)
        print(f"   magnitude {mag:.1f}: found {hits}/{per_mag}")
    return np.array(mags), np.array(out)

rmags, reff = efficiency_on_real()
V50_REAL = depth_50(rmags, reff)
NOTES["V50_real"] = float(V50_REAL)

fig, ax = plt.subplots(figsize=(8, 4.6))
ax.plot(mags, eff * 100, "o--", color=C_AQUA, lw=2, ms=7, label="practice images")
ax.plot(rmags, reff * 100, "o-", color=C_BLUE, lw=2.5, ms=8, label="this patch")
if np.isfinite(V50_REAL):
    ax.axvline(V50_REAL, color=C_ORANGE, lw=2, ls="--")
    ax.text(V50_REAL + 0.05, 60, f"depth of this search:\nmagnitude {V50_REAL:.1f}", color=C_ORANGE, fontweight="bold")
ax.axhline(50, color=GRID, lw=1); ax.legend()
ax.set_xlabel("magnitude of the hidden object (bigger = fainter)"); ax.set_ylabel("found (%)")
ax.set_title("Depth of your search", loc="left"); ax.grid(color=GRID, lw=0.6)
plt.tight_layout(); plt.savefig("B4_depth_real.png", dpi=140); SAVED.append("B4_depth_real.png"); plt.show()

### 4b. The real search
Now we search the patch for real, with no object hidden on purpose.

**How we know what counts as interesting.** Trying thousands of speeds means noise alone throws up thousands of
"5 sigma" bumps. So we run the *same search twice*: once on the real images, and once on the images **flipped in
brightness** (bright becomes dark). Flipping turns every real object into a dip, so anything the flipped run
"finds" is **pure junk with exactly the same noise**. That sets the level a real candidate must beat, with no guesswork.

In [ ]:
SEARCH_GRID_REAL = rate_angle_grid(real_times)
print(f"Trying {len(SEARCH_GRID_REAL)} speed/direction combinations on the patch...")
real_cands = blind_search(real_frames, real_times, SEARCH_GRID_REAL)
print("Now the same search on the images turned upside-down in brightness (pure junk, for comparison)...")
junk_cands = blind_search(-real_frames, real_times, SEARCH_GRID_REAL)
clean_real = remove_static_sky(real_frames)
if len(junk_cands):
    junk_cands["cnn_score"] = [cnn_score(shift_and_stack(clean_real, real_times, r.rate, r.angle), r.x, r.y)
                               for _, r in junk_cands.iterrows()]
    jp = np.sort(junk_cands.loc[junk_cands.cnn_score > 0.5, "snr"].values)
else:
    jp = np.array([])
from scipy.stats import norm
n_spots = int(((~MASK).sum() if MASK is not None else real_frames[0].size) / (2 * psf_sigma_pix()) ** 2)
n_tries = len(SEARCH_GRID_REAL) * max(n_spots, 1)
T_MATHS = float(norm.isf(1.0 / n_tries))                            # level where pure noise gives ~1 false alarm
T_JUNK = float(np.percentile(junk_cands["snr"], 90)) if len(junk_cands) else SNR_THRESHOLD
JUNK_LEVEL = max(T_MATHS, T_JUNK)
NOTES["junk_level"] = JUNK_LEVEL
print(f"   junk run: {len(junk_cands)} candidates, {len(jp)} of them also fooled the CNN")
EXPECTED_JUNK = int((junk_cands["snr"] >= JUNK_LEVEL).sum()) if len(junk_cands) else 0
NOTES["expected_junk"] = EXPECTED_JUNK
print(f"   the maths says {T_MATHS:.1f} sigma, the junk run says {T_JUNK:.1f} sigma "
      f"→ a real candidate must beat {JUNK_LEVEL:.1f} sigma")
print(f"   at that level the junk run itself still has {EXPECTED_JUNK} candidates — that is the number to compare with")
if len(real_cands):
    real_cands["cnn_score"] = [cnn_score(shift_and_stack(clean_real, real_times, r.rate, r.angle), r.x, r.y)
                               for _, r in real_cands.iterrows()]
    real_cands = real_cands.sort_values(["cnn_score", "snr"], ascending=False).reset_index(drop=True)
    real_cands.to_csv("stageB_candidates.csv", index=False); SAVED.append("stageB_candidates.csv")
GOOD = (real_cands[(real_cands.cnn_score > 0.5) & (real_cands.snr >= JUNK_LEVEL)]
        if len(real_cands) else real_cands)
NOTES["n_candidates"] = int(len(real_cands)); NOTES["n_good"] = int(len(GOOD))
print(f"✅ Search done: {len(real_cands)} candidates in total; {len(GOOD)} beat the junk level AND look like "
      f"real objects to the CNN (junk run: {EXPECTED_JUNK} at the same level)")
if len(real_cands): display(real_cands.head(10).round(2))

if len(GOOD):
    n = min(5, len(GOOD))
    fig, axes = plt.subplots(1, n, figsize=(2.6 * n, 3.1))
    for ax, (_, r) in zip(np.atleast_1d(axes), GOOD.head(n).iterrows()):
        st = cut_stamp(shift_and_stack(clean_real, real_times, r.rate, r.angle), r.x, r.y, 41)
        if st is None: st = np.zeros((41, 41), np.float32)
        ax.imshow(np.asarray(st, dtype=float), vmin=-0.2, vmax=1.0); ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(f"{r.snr:.1f}σ, score {r.cnn_score:.2f}\n{r.rate:.1f}\"/day", fontsize=10)
    fig.suptitle("Best candidates (still need checking in other images)", fontweight="bold")
    plt.tight_layout(); plt.savefig("B5_candidates.png", dpi=140); SAVED.append("B5_candidates.png"); plt.show()

## Part 5 — What does this tell us about Planet Nine? ⏱️ seconds
If you uploaded `p9_sky_samples.csv` from Stage A, we can now do the real bookkeeping:
**every possible planet that this search should have seen, and did not, becomes less likely.**

In [ ]:
FEEDBACK = None
if os.path.exists("p9_sky_samples.csv"):
    sky = pd.read_csv("p9_sky_samples.csv")
    half = CUTOUT_ARCMIN / 60 / 2
    d_ra = (np.abs(((sky["ra"] - TARGET_RA + 180) % 360) - 180)) * np.cos(np.radians(sky["dec"]))
    in_patch = (d_ra < half) & (np.abs(sky["dec"] - TARGET_DEC) < half)
    eff_of_mag = lambda m: np.interp(m, rmags, reff, left=reff[0], right=0.0)
    p_det = np.where(in_patch, eff_of_mag(sky["V_mag"].values), 0.0)
    ruled_out = p_det.mean()
    bright_enough = (sky["V_mag"] < V50_REAL).mean() if np.isfinite(V50_REAL) else np.nan
    FEEDBACK = dict(in_patch=float(in_patch.mean()), ruled_out=float(ruled_out), bright_enough=float(bright_enough))
    print(f"Planet Nine possibilities inside this patch: {in_patch.mean()*100:.3f}%")
    print(f"Bright enough for this search anywhere:      {bright_enough*100:.1f}%")
    print(f"→ ruled out by this search:                  {ruled_out*100:.4f}%")
    sky["weight"] = sky.get("weight", 1.0) * (1 - p_det)
    sky.to_csv("p9_sky_samples_updated.csv", index=False); SAVED.append("p9_sky_samples_updated.csv")
    print("✅ Saved p9_sky_samples_updated.csv (use it as the starting point next time)")
else:
    print("ℹ️ No p9_sky_samples.csv uploaded, so this step is skipped.")
    print("   Upload it (folder icon on the left) and run this cell again to see the constraint.")

## ⭐ YOUR RESULTS ⭐

In [ ]:
lines = ["==================== STAGE B RESULTS ====================",
         f"Patch searched: RA {TARGET_RA}, Dec {TARGET_DEC}, size {CUTOUT_ARCMIN} arcmin",
         f"Images: {NOTES['n_images']} " + ("REAL ZTF images" if NOTES["real_images"] else "FAKE images (download failed)")
         + f", spread over {NOTES['baseline_days']:.2f} days", "",
         "1) Does the pipeline work?",
         f"   Practice object (magnitude {TRUTH['mag']}) found at {NOTES['practice_snr']:.1f} sigma: "
         + ("YES" if NOTES["practice_snr"] > SNR_THRESHOLD else "NO"),
         f"   Blind search recovered it without being told the speed: {'YES' if NOTES['blind_found'] else 'NO'}", "",
         "2) The AI (CNN)",
         f"   Accuracy on examples it never saw: {NOTES['cnn_accuracy']:.0%}", "",
         "3) How faint can it see? (50% of objects found)",
         f"   Practice images: magnitude {NOTES['V50_fake']:.1f}",
         f"   This patch:      magnitude {NOTES['V50_real']:.1f}", "",
         "4) The search",
         f"   Candidates found: {NOTES['n_candidates']} in total",
         f"   Junk level (from the flipped-image run): {NOTES.get('junk_level', float('nan')):.1f} sigma",
         f"   Candidates above that level AND passing the CNN: {NOTES['n_good']}",
         f"   Junk run gave {NOTES.get('expected_junk', 0)} at the same level "
         "-> a similar number means nothing real was found",
         "   (survivors are NOT detections: each one must be seen again in separate images)", ""]
if FEEDBACK:
    lines += ["5) What it means for Planet Nine",
              f"   Possibilities inside this patch:   {FEEDBACK['in_patch']*100:.3f}%",
              f"   Bright enough for this depth:      {FEEDBACK['bright_enough']*100:.1f}%",
              f"   Ruled out by this search:          {FEEDBACK['ruled_out']*100:.4f}%", ""]
lines += ["Reminder: ZTF is far too shallow to find Planet Nine (it would be about magnitude 24).",
          "The point of this stage is a working, measured pipeline that deeper images can use.",
          "========================================================="]
summary = "\n".join(lines)
print(summary)
open("stageB_summary.txt", "w").write(summary + "\n"); SAVED.append("stageB_summary.txt")
with zipfile.ZipFile("planet9_stageB_results.zip", "w") as z:
    for fn in SAVED:
        if os.path.exists(fn): z.write(fn)
try:
    from google.colab import files
    files.download("planet9_stageB_results.zip")
    print("\n⬇️ Downloading planet9_stageB_results.zip ...")
except ImportError:
    print("\n📁 planet9_stageB_results.zip is in the same folder as this notebook.")